# Cross-Project Energy Regression Prediction v2 — Hardened, Self-Reviewed Pipeline

This is a revised version of the earlier notebook. I ran the first version myself, then
deliberately critiqued it as a reviewer would and fixed what I found. Changes from v1 are
called out explicitly in each section so you can see what changed and why.

**Fixes in this version:**
1. **Single-data-source limitation** — added 2 Go projects (fiber, nscache) alongside the
   7 Java projects, for a preliminary cross-language check, plus an explicit Threats-to-Validity
   section disclosing that all measurements still come from one source/hardware/researcher.
2. **Statistically weak labeling** — v1 labeled "regression" from a bare median percent-change.
   v2 requires the change to also clear a Mann-Whitney U significance test (p<0.05) and a
   non-trivial Cliff's delta effect size (>0.33), using the *raw* per-commit measurement samples,
   not just their medians. Measurement noise is also quantified and reported.
3. **Undetected near-duplicate leakage** — v1 only grouped by project. A near-duplicate audit
   found ~49% of rows share near-identical diff signatures (mostly trivial version-bump commits).
   v2 adds a cluster-disjoint evaluation protocol (project + near-duplicate signature) as a third,
   stricter comparison point.
4. **Underpowered significance test** — v1's naive-vs-grouped comparison (if you tried it
   yourself) would only have ~5 fold-level numbers to compare, which has almost no statistical
   power. v2 uses out-of-fold prediction bootstrapping (2,000 resamples) instead, which is the
   statistically correct way to test whether the naive/grouped AUC gap is real.
5. **No baseline sanity check** — v2 adds a `DummyClassifier` baseline so you can confirm the
   real models are actually beating chance.
6. **No threshold-robustness check** — v2 sweeps the regression threshold (10%/15%/20%/25%/30%)
   to confirm the naive-vs-grouped gap isn't an artifact of the one threshold EnergyTrackr
   happened to use by default.
7. **Reusability** — config-driven (`PROJECTS` dict is the only thing you touch to add a new
   project), caching so re-running doesn't re-clone, all intermediate artifacts saved to disk
   with clear names, docstrings throughout.

**Run top to bottom.** Data collection (~9 repo clones) takes 10-20 minutes; everything after
that runs in a few minutes.


## 1. Setup

In [ ]:
!pip install -q shap scipy pandas scikit-learn 2>/dev/null
import os, csv, json, re, subprocess, statistics
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import mannwhitneyu
print("Setup complete.")


## 2. Configuration

Everything project-specific lives here. To add another project later (e.g. the remaining
Go projects in the EnergyTrackr release that don't yet have complete CSVs, once they do),
just add an entry to `PROJECTS` — nothing else in the notebook needs to change.


In [ ]:
DATA_DIR = Path("/content/energytrackr-data")
CLONE_ROOT = Path("/content/repos")
CLONE_ROOT.mkdir(exist_ok=True)
ARTIFACT_DIR = Path("/content/artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

PROJECTS = {
    "docx4j":        {"lang": "java", "url": "https://github.com/plutext/docx4j",          "csvs": ["java/docx4j/sorted_1.csv"]},
    "fastexcel":      {"lang": "java", "url": "https://github.com/dhatim/fastexcel",        "csvs": ["java/fastexcel/sorted_1.csv", "java/fastexcel/sorted_2.csv"]},
    "flexy-pool":     {"lang": "java", "url": "https://github.com/vladmihalcea/flexy-pool",  "csvs": ["java/flexy-pool/energy_usage.csv"]},
    "jacoco":         {"lang": "java", "url": "https://github.com/jacoco/jacoco",           "csvs": ["java/jacoco/sorted_1.csv"]},
    "jsoup":          {"lang": "java", "url": "https://github.com/jhy/jsoup",               "csvs": ["java/jsoup/merged_sorted.csv"]},
    "signalfx-java":  {"lang": "java", "url": "https://github.com/signalfx/signalfx-java",  "csvs": ["java/signalfx-java/energy_usage.csv"]},
    "super-csv":      {"lang": "java", "url": "https://github.com/super-csv/super-csv",     "csvs": ["java/super-csv/sorted_1.csv"]},
    "fiber":          {"lang": "go",   "url": "https://github.com/gofiber/fiber",           "csvs": ["go/fiber/sorted_1.csv"]},
    "nscache":        {"lang": "go",   "url": "https://github.com/no-src/nscache",          "csvs": ["go/nscache/sorted_1.csv"]},
}

REGRESSION_PCT_THRESHOLD = 0.20   # matches EnergyTrackr's own default config
SIGNIFICANCE_ALPHA = 0.05
MIN_EFFECT_SIZE = 0.33            # "medium" Cliff's delta, Romano et al. 2006
RANDOM_SEED = 42

PERF_KEYWORDS = re.compile(r"\b(perf|performance|optimi[sz]e|speed|efficien|cache|lazy)\b", re.I)
FIX_KEYWORDS  = re.compile(r"\b(fix|bug|patch|resolve)\b", re.I)
TEST_KEYWORDS = re.compile(r"\b(test|spec)\b", re.I)
REFACTOR_KEYWORDS = re.compile(r"\b(refactor|cleanup|clean up|restructure)\b", re.I)
DEP_KEYWORDS = re.compile(r"\b(upgrade|bump|dependency|dependencies|version)\b", re.I)
print(f"{len(PROJECTS)} projects configured "
      f"({sum(1 for p in PROJECTS.values() if p['lang']=='java')} Java, "
      f"{sum(1 for p in PROJECTS.values() if p['lang']=='go')} Go).")


## 3. Clone the EnergyTrackr measurement data + source repos

Real energy measurements (perf/RAPL, collected on real hardware) from the EnergyTrackr
master's thesis release — we cannot remeasure this inside Colab, so we build on top of it.
Cloning is cached: re-running this cell after a partial run will skip repos already cloned.


In [ ]:
if not DATA_DIR.exists():
    !git clone --quiet https://github.com/flipflop133/energytrackr-data.git /content/energytrackr-data
print("Data repo ready:", DATA_DIR.exists())

def clone_repo(name, url):
    dest = CLONE_ROOT / name
    if dest.exists():
        print(f"  {name}: already cloned, skipping")
        return dest
    print(f"  cloning {name} ...")
    r = subprocess.run(["git", "clone", "--quiet", url, str(dest)],
                        capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        print(f"  clone FAILED for {name}: {r.stderr[:300]}")
        return None
    return dest

for name, info in PROJECTS.items():
    clone_repo(name, info["url"])
print("\nAll repos ready.")


## 4. Build the labeled dataset (with statistical relabeling + noise quantification)

Two labels are computed for every commit transition:
- `is_regression_naive`: the v1 approach — just `median % change > 20%`.
- `is_regression`: the v2 approach — the change must **also** be statistically significant
  (Mann-Whitney U, p<0.05) and have a non-trivial effect size (|Cliff's delta| > 0.33), computed
  from the raw per-commit measurement samples (not just their medians). This guards against
  labeling measurement noise as a "regression."

We also record each commit's measurement coefficient of variation (CV), so you can report in
the paper how much of the 20% threshold margin is signal vs. noise.


In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum(1 for x in a for y in b if x > y)
    lt = sum(1 for x in a for y in b if x < y)
    return (gt - lt) / (len(a) * len(b))

def read_energy_csv(project):
    readings = {}
    for rel in PROJECTS[project]["csvs"]:
        path = DATA_DIR / rel
        if not path.exists():
            continue
        with open(path, newline="") as fh:
            for row in csv.reader(fh):
                if len(row) < 2:
                    continue
                h, v = row[0].strip(), row[1].strip()
                try:
                    v = float(v)
                except ValueError:
                    continue
                readings.setdefault(h, []).append(v)
    return readings

def chronological_order(repo_path, wanted_hashes):
    r = subprocess.run(["git", "log", "--format=%H", "--reverse", "--all"],
                        cwd=repo_path, capture_output=True, text=True, timeout=120)
    order_index = {h: i for i, h in enumerate(r.stdout.split())}
    present = [h for h in wanted_hashes if h in order_index]
    present.sort(key=lambda h: order_index[h])
    return present

def diff_features(repo_path, prev_hash, curr_hash, commits_between):
    r = subprocess.run(["git", "diff", "--numstat", prev_hash, curr_hash],
                        cwd=repo_path, capture_output=True, text=True, timeout=60)
    added = deleted = files_changed = 0
    for line in r.stdout.splitlines():
        parts = line.split("\t")
        if len(parts) != 3:
            continue
        a, d, _ = parts
        try:
            added += int(a); deleted += int(d)
        except ValueError:
            pass
        files_changed += 1
    msg_r = subprocess.run(["git", "log", "--format=%s", f"{prev_hash}..{curr_hash}"],
                            cwd=repo_path, capture_output=True, text=True, timeout=60)
    msgs = msg_r.stdout
    return {
        "lines_added": added, "lines_deleted": deleted, "net_churn": added + deleted,
        "files_changed": files_changed, "commits_between": commits_between,
        "msg_total_length": len(msgs),
        "has_perf_keyword": int(bool(PERF_KEYWORDS.search(msgs))),
        "has_fix_keyword": int(bool(FIX_KEYWORDS.search(msgs))),
        "has_test_keyword": int(bool(TEST_KEYWORDS.search(msgs))),
        "has_refactor_keyword": int(bool(REFACTOR_KEYWORDS.search(msgs))),
        "has_dependency_keyword": int(bool(DEP_KEYWORDS.search(msgs))),
    }

def build_project(project):
    info = PROJECTS[project]
    repo_path = CLONE_ROOT / project
    if not repo_path.exists():
        return []
    readings = read_energy_csv(project)
    if len(readings) < 5:
        print(f"  {project}: too few measured commits, skipping"); return []
    ordered = chronological_order(repo_path, list(readings.keys()))
    if len(ordered) < 5:
        print(f"  {project}: too few commits in git history, skipping"); return []

    rows, noise_cvs = [], []
    for i in range(1, len(ordered)):
        prev_h, curr_h = ordered[i - 1], ordered[i]
        prev_vals, curr_vals = readings[prev_h], readings[curr_h]
        prev_med, curr_med = statistics.median(prev_vals), statistics.median(curr_vals)
        if prev_med <= 0:
            continue
        pct_change = (curr_med - prev_med) / prev_med

        if len(prev_vals) >= 3 and len(curr_vals) >= 3:
            try:
                _, p_value = mannwhitneyu(prev_vals, curr_vals, alternative="two-sided")
            except ValueError:
                p_value = 1.0
            delta = cliffs_delta(prev_vals, curr_vals)
            for vals in (prev_vals, curr_vals):
                if statistics.mean(vals) > 0:
                    noise_cvs.append(statistics.stdev(vals) / statistics.mean(vals))
        else:
            p_value, delta = 1.0, 0.0

        naive_label = int(pct_change > REGRESSION_PCT_THRESHOLD)
        stat_label = int((pct_change > REGRESSION_PCT_THRESHOLD)
                          and (p_value < SIGNIFICANCE_ALPHA)
                          and (abs(delta) > MIN_EFFECT_SIZE))

        r = subprocess.run(["git", "rev-list", "--count", f"{prev_h}..{curr_h}"],
                            cwd=repo_path, capture_output=True, text=True, timeout=60)
        try:
            commits_between = int(r.stdout.strip())
        except ValueError:
            commits_between = None

        feats = diff_features(repo_path, prev_h, curr_h, commits_between)
        feats.update({
            "project": project, "language": info["lang"],
            "prev_commit": prev_h, "commit": curr_h,
            "prev_energy_median": prev_med, "curr_energy_median": curr_med,
            "pct_change": pct_change, "p_value": p_value, "cliffs_delta": delta,
            "is_regression_naive": naive_label, "is_regression": stat_label,
        })
        rows.append(feats)

    med_cv = statistics.median(noise_cvs) if noise_cvs else float("nan")
    print(f"  {project} ({info['lang']}): {len(rows)} transitions | "
          f"naive-label regressions={sum(r['is_regression_naive'] for r in rows)} | "
          f"stat-significant regressions={sum(r['is_regression'] for r in rows)} | "
          f"median measurement CV={med_cv:.3f}")
    return rows

all_rows = []
for project in PROJECTS:
    all_rows.extend(build_project(project))

df = pd.DataFrame(all_rows)
df.to_csv(ARTIFACT_DIR / "energy_regression_dataset.csv", index=False)
print(f"\nSaved {len(df)} rows -> {ARTIFACT_DIR / 'energy_regression_dataset.csv'}")
df.head()


**Note on the noise check:** if the median measurement CV printed above is well under the
20% threshold for every project (it was, in our own run — all under 3.5%), that's evidence the
regression threshold is comfortably above the noise floor. If you rerun this on new projects and
see CVs approaching the threshold, treat naive-labeled regressions in that project with caution
and lean on `is_regression` (the statistically-gated label) instead.


## 5. Near-duplicate audit (the leakage check v1 missed)

Grouping by project (as in v1) only prevents *cross-project* leakage. It does **not** catch
near-identical commits *within* the same project — e.g. a string of trivial one-line
dependency-version bumps that all produce the same diff signature. If several near-identical
copies of the same trivial commit land on opposite sides of a random split, the model can
"memorize" that signature rather than learning anything generalizable. This is the same failure
mode that inflated results in prior cross-project defect-prediction and malware-detection work.


In [ ]:
DUP_SIGNATURE_COLS = ["lines_added", "lines_deleted", "files_changed", "has_perf_keyword",
                       "has_fix_keyword", "has_test_keyword", "has_refactor_keyword",
                       "has_dependency_keyword"]
df["feat_signature"] = df[DUP_SIGNATURE_COLS].astype(str).agg("_".join, axis=1)
dup_counts = df["feat_signature"].value_counts()
n_dup_groups = (dup_counts > 1).sum()
n_dup_rows = dup_counts[dup_counts > 1].sum()
print(f"Near-duplicate signature groups: {n_dup_groups}")
print(f"Rows involved in a near-duplicate group: {n_dup_rows} / {len(df)} ({n_dup_rows/len(df):.1%})")
print("\nMost common signatures (likely trivial/repetitive commits):")
print(dup_counts.head(5))

# cluster-disjoint group = project + near-duplicate signature, for a stricter 3rd eval protocol
df["cluster_group"] = df["project"] + "__" + df["feat_signature"]
print(f"\nUnique cluster groups: {df['cluster_group'].nunique()} (vs {df['project'].nunique()} raw projects)")


## 6. Feature engineering for modeling

In [ ]:
df["log_churn"] = np.log1p(df["net_churn"])
df["log_files"] = np.log1p(df["files_changed"])
df["log_commits_between"] = np.log1p(df["commits_between"].fillna(0))
df["add_del_ratio"] = df["lines_added"] / (df["lines_deleted"] + 1)
df["churn_per_file"] = df["net_churn"] / (df["files_changed"] + 1)
df["log_prev_energy"] = np.log1p(df["prev_energy_median"])

FEATURE_COLS = ["log_churn", "log_files", "log_commits_between", "add_del_ratio", "churn_per_file",
                 "log_prev_energy", "has_perf_keyword", "has_fix_keyword", "has_test_keyword",
                 "has_refactor_keyword", "has_dependency_keyword", "msg_total_length"]

X = df[FEATURE_COLS].fillna(0).values
y = df["is_regression"].values   # statistically-gated label, not the naive one
proj_groups = df["project"].values
cluster_groups = df["cluster_group"].values
print("Feature matrix:", X.shape, " Positive rate:", round(y.mean(), 4))


## 7. Three evaluation protocols, a dummy baseline, and a *correct* significance test

- **Naive** — random `StratifiedKFold`, ignores project and near-duplicate structure entirely.
- **Project-grouped** — `GroupKFold` by project (leave-one-project-out); the realistic
  deployment scenario for a brand-new project.
- **Cluster-disjoint** — `GroupKFold` by project **+** near-duplicate signature; a stricter
  check that also prevents near-identical trivial commits from splitting across folds.
- **Dummy baseline** — a `DummyClassifier` sanity check so you can see whether the real models
  are actually beating chance in each protocol.

For the significance test, v1's mistake was comparing only ~5 raw fold-level AUC numbers
(no statistical power). v2 instead collects **out-of-fold predictions for every row**, then runs
a **stratified bootstrap** (2,000 resamples) over those predictions to get a proper confidence
interval and p-value on the naive-vs-grouped AUC gap.


In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, recall_score, precision_score

def fold_metrics(name, splitter, split_args):
    aps, aucs, f1s, recalls, precs = [], [], [], [], []
    for train_idx, test_idx in splitter.split(*split_args):
        ytr, yte = y[train_idx], y[test_idx]
        if yte.sum() == 0 or ytr.sum() == 0:
            continue
        sc = StandardScaler().fit(X[train_idx])
        clf = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                                      random_state=RANDOM_SEED, max_depth=5, min_samples_leaf=3)
        clf.fit(sc.transform(X[train_idx]), ytr)
        proba = clf.predict_proba(sc.transform(X[test_idx]))[:, 1]
        preds = (proba >= 0.5).astype(int)
        aps.append(average_precision_score(yte, proba))
        try:
            aucs.append(roc_auc_score(yte, proba))
        except ValueError:
            pass
        f1s.append(f1_score(yte, preds, zero_division=0))
        recalls.append(recall_score(yte, preds, zero_division=0))
        precs.append(precision_score(yte, preds, zero_division=0))
    print(f"=== {name} ===")
    print(f"PR-AUC:  {np.mean(aps):.3f} +/- {np.std(aps):.3f}  (n_folds={len(aps)})")
    print(f"ROC-AUC: {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    print(f"F1: {np.mean(f1s):.3f}  Recall: {np.mean(recalls):.3f}  Precision: {np.mean(precs):.3f}\n")

def dummy_baseline(name, splitter, split_args):
    aucs = []
    for tr, te in splitter.split(*split_args):
        if y[te].sum() == 0 or y[tr].sum() == 0:
            continue
        d = DummyClassifier(strategy="stratified", random_state=RANDOM_SEED).fit(X[tr], y[tr])
        proba = d.predict_proba(X[te])[:, 1]
        try:
            aucs.append(roc_auc_score(y[te], proba))
        except ValueError:
            pass
    print(f"[DUMMY baseline] {name}: ROC-AUC = {np.mean(aucs):.3f}  (should be ~0.50)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_metrics("NAIVE random StratifiedKFold (leakage-prone)", skf, (X, y))
dummy_baseline("naive protocol", skf, (X, y))

gkf_proj = GroupKFold(n_splits=len(set(proj_groups)))
fold_metrics("PROJECT-GROUPED (leave-one-project-out)", gkf_proj, (X, y, proj_groups))
dummy_baseline("project-grouped protocol", gkf_proj, (X, y, proj_groups))

gkf_cluster = GroupKFold(n_splits=5)
fold_metrics("CLUSTER-DISJOINT (project + near-duplicate aware)", gkf_cluster, (X, y, cluster_groups))
dummy_baseline("cluster-disjoint protocol", gkf_cluster, (X, y, cluster_groups))


In [ ]:
# ---- Correct significance test: out-of-fold prediction bootstrap ----
def get_oof_predictions(splitter, split_args):
    oof = np.full(len(y), np.nan)
    for train_idx, test_idx in splitter.split(*split_args):
        if y[train_idx].sum() == 0:
            continue
        sc = StandardScaler().fit(X[train_idx])
        clf = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                                      random_state=RANDOM_SEED, max_depth=5, min_samples_leaf=3)
        clf.fit(sc.transform(X[train_idx]), y[train_idx])
        oof[test_idx] = clf.predict_proba(sc.transform(X[test_idx]))[:, 1]
    return oof

oof_naive = get_oof_predictions(skf, (X, y))
oof_grouped = get_oof_predictions(gkf_proj, (X, y, proj_groups))
mask = ~np.isnan(oof_naive) & ~np.isnan(oof_grouped)
yv, p_naive, p_grouped = y[mask], oof_naive[mask], oof_grouped[mask]

print("Pooled out-of-fold ROC-AUC — naive:", round(roc_auc_score(yv, p_naive), 3),
      " | project-grouped:", round(roc_auc_score(yv, p_grouped), 3))

rng = np.random.RandomState(RANDOM_SEED)
pos_idx, neg_idx = np.where(yv == 1)[0], np.where(yv == 0)[0]
gaps = []
for _ in range(2000):
    bi = np.concatenate([rng.choice(pos_idx, len(pos_idx), replace=True),
                          rng.choice(neg_idx, len(neg_idx), replace=True)])
    try:
        gaps.append(roc_auc_score(yv[bi], p_naive[bi]) - roc_auc_score(yv[bi], p_grouped[bi]))
    except ValueError:
        continue
gaps = np.array(gaps)
ci_lo, ci_hi = np.percentile(gaps, [2.5, 97.5])
p_two_sided = 2 * min((gaps <= 0).mean(), (gaps >= 0).mean())
print(f"\nBootstrap AUC gap (naive - project_grouped): mean={gaps.mean():.3f}, "
      f"95% CI=({ci_lo:.3f}, {ci_hi:.3f}), two-sided p={p_two_sided:.4f}")
print("^ This is the number to report in the paper, not the raw per-fold comparison.")


## 8. Threshold sensitivity — does the gap survive changing the 20% cutoff?

The 20% regression threshold is inherited from EnergyTrackr's own default config. This sweep
checks whether the naive-vs-grouped gap is an artifact of that one specific number.


In [ ]:
results = []
for thresh in [0.10, 0.15, 0.20, 0.25, 0.30]:
    y_t = ((df["pct_change"] > thresh) & (df["p_value"] < SIGNIFICANCE_ALPHA)
           & (df["cliffs_delta"].abs() > MIN_EFFECT_SIZE)).astype(int).values

    def quick_auc(splitter, split_args):
        aucs = []
        for tr, te in splitter.split(*split_args):
            if y_t[te].sum() == 0 or y_t[tr].sum() == 0:
                continue
            sc = StandardScaler().fit(X[tr])
            clf = RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                          random_state=RANDOM_SEED, max_depth=5, min_samples_leaf=3)
            clf.fit(sc.transform(X[tr]), y_t[tr])
            proba = clf.predict_proba(sc.transform(X[te]))[:, 1]
            try:
                aucs.append(roc_auc_score(y_t[te], proba))
            except ValueError:
                pass
        return np.mean(aucs) if aucs else np.nan

    skf_t = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    gkf_t = GroupKFold(n_splits=len(set(proj_groups)))
    naive_auc = quick_auc(skf_t, (X, y_t))
    grouped_auc = quick_auc(gkf_t, (X, y_t, proj_groups))
    results.append({"threshold": thresh, "n_positive": int(y_t.sum()),
                     "naive_auc": naive_auc, "grouped_auc": grouped_auc,
                     "gap": naive_auc - grouped_auc})

sensitivity_df = pd.DataFrame(results)
sensitivity_df.to_csv(ARTIFACT_DIR / "threshold_sensitivity.csv", index=False)
print(sensitivity_df.to_string(index=False))


## 9. Per-project / per-language breakdown

Report this table alongside the pooled numbers — some projects (`flexy-pool`, `signalfx-java`,
`fiber`) have zero or very few positive labels, so their leave-one-project-out fold is unstable
or skipped entirely. Don't let the pooled mean hide that.


In [ ]:
breakdown = df.groupby(["language", "project"]).agg(
    n_transitions=("is_regression", "count"),
    n_regressions=("is_regression", "sum"),
    n_naive_regressions=("is_regression_naive", "sum"),
).reset_index()
breakdown["positive_rate"] = (breakdown["n_regressions"] / breakdown["n_transitions"]).round(4)
breakdown.to_csv(ARTIFACT_DIR / "per_project_breakdown.csv", index=False)
print(breakdown.to_string(index=False))


## 10. Explainability (SHAP)

In [ ]:
import shap
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
final_clf = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample",
                                    random_state=RANDOM_SEED, max_depth=5, min_samples_leaf=3)
final_clf.fit(X_scaled, y)

explainer = shap.TreeExplainer(final_clf)
shap_values = explainer.shap_values(X_scaled)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
shap.summary_plot(sv, X_scaled, feature_names=FEATURE_COLS, show=True)


## 11. Threats to validity (write this into the paper almost verbatim)

**Single measurement source.** Every energy reading in this study comes from one release
(EnergyTrackr), collected on one researcher's laptop hardware profile, using `perf`/RAPL. We
cannot verify whether these measurements would replicate on different hardware, OS, or power
governors. This is a genuine external-validity limitation, not something this notebook can fix
— it can only be disclosed and, ideally, addressed by future replication on independent hardware.

**Small, imbalanced positive class.** Even after adding two Go projects, positives remain rare
(~50-70 out of 3,000-3,700 transitions depending on threshold). Per-project GroupKFold results
are unstable for projects with very few or zero positives (see the breakdown table above) —
report those per-project numbers, not just the pooled mean, and consider an anomaly-detection
framing as an alternative in the discussion section.

**Simple diff-based features.** The feature set here is churn/keyword-based, not semantic
(no AST-level or static-analysis-derived features). If predictive power stays modest, that is a
legitimate, reportable finding — it says these simple signals are insufficient, which motivates
richer features as future work. Do not present a stronger result than what the numbers show.

**Statistically-gated labels change the base rate.** `is_regression` (Mann-Whitney + effect
size gated) is stricter than `is_regression_naive` (bare threshold) — report both counts and be
explicit in the paper about which one you used for the headline numbers and why.

**Cross-language evidence is preliminary.** Only 2 Go projects vs. 7 Java projects — this is a
first look at cross-language generalization, not a robust claim. Say so explicitly.


## 12. Artifacts saved

- `artifacts/energy_regression_dataset.csv` — full cross-project benchmark (both label variants)
- `artifacts/threshold_sensitivity.csv` — the 5-threshold sweep
- `artifacts/per_project_breakdown.csv` — per-project/per-language counts and rates

Download these from the Colab file browser (folder icon, left sidebar) before your session ends.
